# Custosell Fiscal Engine  -  Architecture + Implementation Plan

**Owner:** Mike (Backend Orchestrator) | **Team:** Custospark Product Team | **For:** Oscar

**Date:** 2026-09-25 | **Status:** DRAFT for approval | **Location:** `Backend/docs/fiscal-engine-architecture.ipynb`

> Problem #1: Mandatory E-Invoicing (URA EFRIS / KRA eTIMS)  -  Middleware + POS Hardware. Rating ★★★★★ (5/5).
>
> **Decision locked (2026-09-25):** Modular monolith inside Custosell, NOT a separate app. One codebase, two modes: `Full POS` + `Fiscal Headless`. Thin on-prem edge agent for legacy ERPs. Hardware in Phase 3.

How to use this notebook:
1. Read top-to-bottom for architecture.
2. Execute the checklist cells to track progress.
3. Each Phase ends with a Vera gate + Oscar approval before the next Phase starts.
4. Deployment always follows `/DEPLOYMENT.md` (never wipe prod, additive migrations only, `migrate --force`).

## 1. Problem recap (why Priority #1)

- URA EFRIS + KRA eTIMS mandate real-time e-invoices / fiscal receipts. Non-compliance = closure, bank freeze, UGX 6M-30M penalty per transaction.
- 48% of mid-large taxpayers stuck: downtime, stock mismatches, double-entry on URA portal.
- Legacy ERPs (Tally, Sage, QuickBooks Desktop) can't sign JSON in real time. SAP/Oracle custom integration > $40k  -  unaffordable locally.
- Third-party handheld EFRIS gadgets crash + lose offline cache.

**Custospark opportunity:** `Custosell Fiscal Engine`  -  plug-and-play middleware:
(a) local AES-256 cache + offline→online queue with fiscal signatures,
(b) stock reconciliation (physical vs URA virtual stock),
(c) rugged Android/Linux POS with embedded firmware.

**Money:** Setup UGX 45-75M + annual UGX 18-36M + hardware UGX 1.85M/unit. 6-8 distributors = UGX 1B/yr target.
Targets: Mukwano, Kakira/Madhvani, Roofings, CiplaQCIL, Game/Carrefour.

## 2. What Custosell already has (60% built  -  verified 2026-09-25)

### Backend  -  Laravel 12 (`Backend/`)
- `app/Services/Efris/EfrisService.php` + `EfrisClient.php` + `EfrisServiceInterface.php`  -  skeleton engine, never-blocks-checkout.
- `app/Jobs/FiscalizeSaleJob.php`, `FiscalizeInvoiceJob.php`  -  5 tries, backoff `[60,120,300,600,1800]`. This IS the offline→online queue.
- `config/efris.php`  -  `EFRIS_ENABLED`, `country`, `scope.pos_sales/sales_invoices`, `offline=sync_later`, sandbox vs `https://efrisws.ura.go.ug`.
- `Sale.php`, `Invoice.php` already carry `fiscal_status, fiscal_fdn, fiscal_qr, fiscal_verification_code, fiscal_payload/response, fiscalized_at, fiscal_last_error`.
- `SaleService.php:203` calls `efrisService->fiscalizeSale()` inside DB transaction.
- `app/Support/TaxEngine.php`  -  `vat_registered`, `standard/exempt/zero_rated`, inclusive/exclusive. Mirrored in FE `taxEngine.ts`.
- `TaxJurisdictions.php` + FE `taxJurisdictions.ts`  -  UG 18% URA, KE 16% KRA, TZ TRA, RW RRA mapped.
- `Location`, `LocationProduct`, `StockMovement`, `SyncService.php`  -  per-branch ledger + offline push/pull.

### Frontend  -  Electron + React + TS (`Frontend/`)
- `offline/core/offlineDb.ts` (IndexedDB + memory fallback), `sync/mutationQueue.ts`, `syncCoordinator.ts`  -  POS sells offline, replays `/sales` later.
- `shared/utils/taxEngine.ts` mirrors backend tax math for POS preview.
- `docs/compliance/efris-setup.md`  -  credential setup guide exists.

### Gaps (what's NOT real yet)
1. `EfrisClient` is basic-auth + simplified JSON  -  no real URA T-code AES/RSA signing.
2. Credentials are global `.env`  -  need per-`business_id + location_id` vault (Roofings 10 depots can't share one deviceNo).
3. No URA virtual-stock import / variance screen.
4. No Tally/Sage/QuickBooks/Excel bridge.
5. `isEnabled()` returns false unless `country==UG`  -  need driver interface for KE eTIMS.
6. No rugged hardware firmware  -  but same Electron build can target ARM first.

## 3. Architecture decision  -  modular monolith (no separate app)

**Chosen:** Single Laravel codebase, two runtime modes + thin edge agent.

```
                ┌──────────────────────────────────────────────┐
                │           CUSTOSELL BACKEND (Laravel 12)       │
                │  Modular monolith  -  one deploy, two modes     │
                │                                              │
                │  SaleService / InvoiceService / TaxEngine    │
                │  Location / StockMovement ledger             │
                │  ┌────────────────────────────────────────┐  │
                │  │ FISCAL BOUNDED CONTEXT                 │  │
                │  │ FiscalDriverInterface                  │  │
                │  │  ├─ UgEfrisDriver (URA S2S + crypto)   │  │
                │  │  └─ KeEtimsDriver (KRA, Phase 4)       │  │
                │  │ EfrisService / EfrisClient / Crypto    │  │
                │  │ FiscalizeSaleJob / FiscalizeInvoiceJob │  │
                │  │ fiscal_credentials / fiscal_documents  │  │
                │  └────────────────────────────────────────┘  │
                │  Adapter layer: Excel/Tally/Sage/QB import   │
                └──────────▲───────────────────────▲───────────┘
                           │                       │
        Mode A: Full POS │                       │ Mode B: Headless API
   Electron POS + Inventory│                       │ legacy ERP → POST /fiscal/*
                           │                       │ returns FDN/QR
              ┌────────────┴─────┐        ┌────────┴─────────┐
              │ Thin EDGE AGENT  │        │  URA / KRA APIs  │
              │ (on-prem, 200    │        │  sandbox → prod  │
              │  lines, file-    │        └──────────────────┘
              │  watcher + AES   │
              │  cache + retry)  │
              └──────────────────┘
```

**Why not separate app now:** doubles auth, branches, sync contracts, Vera gates, deploys for zero extra revenue. Sale + stock + fiscal must stay in one DB transaction or URA penalties hit on drift.

**Split later only if:** >50 enterprise tenants need independent fiscal scaling, crypto regimes diverge hard, or a client demands on-prem-only vault. Interfaces + `bootstrap/providers.php` bindings make extraction clean.

## 4. Data model

### 4.1 Reuse (no changes)
- `businesses`  -  has `tax_id, tax_regime, jurisdiction, default_vat_rate, prices_include_tax`. Add TIN validation only.
- `locations`  -  branch = URA branch. `is_default` exists.
- `products`  -  `sku, barcode, tax_class, tax_percentage, unit_price/wholesale_price`. Add `efris_commodity_code, efris_unit_code` (nullable) later.
- `sales (receipt_number)`, `sale_items`, `invoices`, `invoice_items`  -  already have fiscal columns.
- `stock_movements (type sale/return, quantity_change, stock_before/after, location_id)`  -  ledger source of truth.
- `location_products (location_id, product_id, stock_quantity)`  -  per-branch physical stock.

### 4.2 New tables (additive migrations only  -  never edit old ones)

| Table | Key columns | Purpose |
|---|---|---|
| `fiscal_credentials` | `business_id, location_id UNIQUE, country, tin, device_no, branch_id, api_username, api_password_encrypted, private_key_ref, environment, is_active` | Per-depot URA identity. Encrypted at rest. |
| `fiscal_documents` | `business_id, location_id, sale_id?, invoice_id?, country, document_type, payload_json, signature, status[pending/fiscalized/failed], fdn, qr, verification_code, response_json, retry_count, last_error, fiscalized_at` | Idempotent fiscal log. One row per fiscal attempt. Unique on `(sale_id)` / `(invoice_id)` where not null. |
| `fiscal_stock_snapshots` | `business_id, location_id, product_id, ura_virtual_qty, physical_qty, variance, synced_at, resolved_at` | Stock recon. Import URA virtual stock, diff vs `location_products`. |
| `fiscal_import_batches` | `business_id, source[tallly/sage/quickbooks/excel], file_ref, total_rows, ok_rows, failed_rows, errors_json, created_by` | Legacy ERP bridge audit. |

Entity order respects FKs: `fiscal_credentials` (after Business+Location) → `fiscal_documents` (after Sale+Invoice) → `fiscal_stock_snapshots` (after Product+Location) → `fiscal_import_batches`.
Each entity = 12 files per `Backend/AGENTS.md`: migration, model, repo interface+impl, service interface+impl, request, resource, collection, controller, `routes/api/v1/*.php`, provider binding.

## 5. Backend components

```
app/Services/Fiscal/
  FiscalDriverInterface.php      # fiscalizeSale(), fiscalizeInvoice(), queryStock(), submitStock()
  UgEfrisDriver.php              # wraps EfrisService + real crypto (Phase 1)
  KeEtimsDriver.php              # stub now, full in Phase 4
  FiscalCryptoService.php        # AES-256 payload encrypt + RSA/T-code sign + verify
  FiscalCredentialService.php    # per-location vault read (decrypt on use, never log)
  FiscalDocumentService.php      # idempotent log + retry_count + status machine
  StockReconciliationService.php # URA import → diff → adjust proposal
  adapters/
    ExcelImportService.php       # phpspreadsheet, Phase 2 first
    TallyAdapter.php             # CSV/XML sales export → SaleService::create()
    SageAdapter.php / QuickBooksAdapter.php
```

- Keep `EfrisService` as the `UgEfrisDriver` core  -  extend, don't fork. Add `forceSync` path for jobs, `sync_later` path for checkout.
- `EfrisClient::submitInvoice()`  -  replace basic-auth stub with URA S2S envelope + signed `data` field + FDN/QR/`antiFakeCode` parsing (already partially there).
- Jobs stay: `FiscalizeSaleJob`, `FiscalizeInvoiceJob` with 5 tries + backoff. On exhaust → `fiscal_status=failed`, loud log + Settings badge.
- SOLID: every repo/service gets interface, binding in dedicated `FiscalServiceProvider`, registered in `bootstrap/providers.php`. No file >500 lines.
- API: `POST /api/v1/fiscal/sales`, `/fiscal/invoices`, `GET /fiscal/status`, `GET /fiscal/documents`, `POST /fiscal/retry/:id`, `GET/POST /fiscal/stock-reconciliation`, `POST /fiscal/imports`. Headless mode = same routes, POS UI flag off.

## 6. Core flows (failure states mandatory)

### 6.1 Sale → fiscal (never blocks till)
1. POS `POST /sales` → `SaleService::create()` in DB transaction → stock decrement + `StockMovement`.
2. `EfrisService::fiscalizeSale($sale)`  -  if `EFRIS_ENABLED=false` or scope off → return sale untouched.
3. If offline (`fsockopen` to URA fails) → `fiscal_status=pending` + `FiscalizeSaleJob::dispatch()` → return 201 to till in <2s.
4. If online → build payload (sellerTin/deviceNo/branchId per `fiscal_credentials.location_id`) → encrypt+sign → `submitInvoice()` → store FDN/QR/`antiFakeCode` → `fiscalized`.
5. On throw → `failed` + `last_error` + job retry in 2 min. After 5 fails → stay `failed`, surface in Settings + receipt reprint queue. Duplicate submit guarded by `fiscal_documents.sale_id UNIQUE` + `if fiscalized return`.

### 6.2 Stock reconciliation (the Kikuubo fix)
1. Nightly + on-demand `queryStock()` pulls URA virtual qty per `(branch, sku)` → `fiscal_stock_snapshots`.
2. Diff vs `location_products.stock_quantity`. Variance table: match / over / short.
3. Clerk proposes adjustment (counts, damage, transfer) → manager approves → `StockMovement(type=adjustment)` + `submitStock()` to URA. Never auto-push without approval.

### 6.3 Legacy ERP bridge
1. Distributor exports Tally Daybook / Sage CSV / Excel template → drops in watched folder or uploads to `/fiscal/imports`.
2. `fiscal_import_batches` row created → row-by-row validate (TIN, VAT math via `TaxEngine`, branch map) → `SaleService::create()` or Invoice equivalent.
3. Each created sale flows through 6.1. Batch report: ok/failed rows + downloadable errors. No partial silent loss.

Validation / auth / duplicate / rollback / retry answers required on every flow per AGENTS.md §10.

## 7. Frontend (Electron + React)

- **POS sale screen:** fiscal badge (`none/pending/fiscalized/failed`), non-blocking toast on `pending`, FDN + QR on receipt reprint. Reuse `ReceiptBusinessHeader`, `receiptTotals`, `taxEngine.computeSaleTax` for preview.
- **Settings → Fiscal:** `publicStatus()` only (enabled/configured/country/mode/environment, never secrets). Per-branch credential form (admin-only), test-connection button, retry-failed button.
- **Stock → Reconciliation tab:** variance table (physical vs URA virtual), filter by branch, approve-adjust flow, audit trail.
- **Imports page:** upload Excel/CSV, column-map, batch progress, error download.
- **Offline:** sales already queue in `mutationQueue` (IndexedDB). Fiscal retry is server-side job  -  no FE polling loop. `syncPendingIfOnline` drains sales first, fiscal follows.
- File size ≤500 lines; Vera `npm run vera:fast` + `tsc --noEmit -p tsconfig.app.json` every module. Docs under `Frontend/docs/` + ADR per feature.

## 8. Multi-country adapter matrix (95% reuse)

| Jurisdiction | Authority | VAT | API style | Driver | Phase |
|---|---|---|---|---|---|
| UG | URA EFRIS | 18% | REST/JSON S2S + signing | `UgEfrisDriver` | Phase 1 (full) |
| KE | KRA eTIMS | 16% | REST/JSON | `KeEtimsDriver` | Phase 4 |
| TZ | TRA EFDMS | 18% | REST/JSON | stub | Phase 4 |
| RW | RRA EBM v2 | 18% | REST/SOAP | stub | Phase 4 |
| ZM/GH | ZRA/GRA |  -  | REST/JSON | stub | backlog |

Only payload adapter + auth + FDN field mapping changes. `TaxEngine` rate comes from `Business.default_vat_rate` + `TaxJurisdictions`, never hardcoded.

## 9. Hardware (Phase 3  -  after software pays)

- **V1:** White-label Android (Sunmi-class) + our Electron POS compiled for ARM. Thermal print: business header, FDN, verification code, QR, totals. No custom firmware yet.
- **V2 (only after 6-8 paying distributors):** Rugged Linux terminal + `Custosell Embedded` firmware, local AES-256 spool (7-day offline), auto-replay. UGX 1.85M/unit.
- Kill criteria: if third-party terminal + our app passes 30-day pilot with zero cache-loss, defer custom hardware.

## 10. Security + compliance

- Secrets in `Backend/.env` + `fiscal_credentials` encrypted columns only. Never commit TIN/passwords/keys. Never log payload secrets; log `sale_id, branch, status, http_code` only.
- `EFRIS_PRIVATE_KEY_PATH` on disk with 0600, prod key via env mount, never in git.
- RBAC via `spatie/laravel-permission`: only Admin/Finance can view credentials, retry fiscal, approve stock adjustments.
- Audit: `fiscal_documents` + `PlatformAuditLog` for every retry/adjust. Receipts show FDN + QR for URA verification.
- Sandbox (`efristest.ura.go.ug`) for all dev/staging; prod switch requires Oscar sign-off + DB backup.

## 11. Implementation plan  -  start to finish (follow in order)

> Rule: Sage plans → Blue designs (3+ files) → Rex codes (reads existing first) → Vera gates → Quill docs → Mike reports to Oscar. One Phase at a time.

### Phase 0  -  Validate (Weeks 1-2)  -  $0 build
- [ ] P0.1 Obtain URA sandbox TIN/device/branch + API user (docs/compliance/efris-setup.md). Confirm real signing spec (T-code, AES/RSA).
- [ ] P0.2 Spike: single sandbox `saveSales` call from `EfrisClient` with real signature. Record raw req/res in `docs/forensic/` (redacted).
- [ ] P0.3 Interview 2 distributors (Mukwano/Roofings contact): confirm stock-mismatch + penalty pain, Excel/Tally formats, branch count.
- [ ] P0.4 ADR: `docs/adr/YYYY-MM-DD-fiscal-headless-mode.md` locking modular-monolith decision.
- **Gate:** Oscar approves Phase 1 scope. Kill if URA sandbox inaccessible >3 weeks.

### Phase 1  -  Harden Fiscal Core (Weeks 3-8)  -  the enterprise MVP
- [ ] P1.1 Migration+models: `fiscal_credentials`, `fiscal_documents` (12-file entity each per AGENTS.md). Provider `FiscalServiceProvider` in `bootstrap/providers.php`.
- [ ] P1.2 `FiscalCryptoService` (encrypt/sign/verify) + `FiscalCredentialService` (per-location decrypt). Unit tests with fake keys.
- [ ] P1.3 Upgrade `EfrisClient::submitInvoice` to real envelope + FDN/QR/`antiFakeCode` parsing. Keep `sync_later` + jobs intact.
- [ ] P1.4 `FiscalDriverInterface` + extract `UgEfrisDriver` from `EfrisService`. `KeEtimsDriver` stub returning `not_supported`.
- [ ] P1.5 Headless routes: `routes/api/v1/fiscal.php` (`sales, invoices, status, documents, retry`). Scope + auth + rate-limit.
- [ ] P1.6 FE: fiscal badge on POS + receipt FDN/QR + Settings→Fiscal status + retry button.
- [ ] P1.7 Pilot: 1 live branch, sandbox → prod with backup. 30-day soak: % fiscalized <5min, zero lost sales.
- **Gate:** `composer vera:fast` + `vera:extended` (migrate --pretend + filtered tests) green. Quill updates `docs/entities.md` + `docs/decisions.md`.

### Phase 2  -  Middleware + Stock Recon (Weeks 9-14)  -  where UGX 45-75M comes from
- [ ] P2.1 `fiscal_stock_snapshots` + `StockReconciliationService` (URA pull → diff vs `location_products`). FE variance tab + approve flow.
- [ ] P2.2 `fiscal_import_batches` + `ExcelImportService` (template + validation + batch report). Pilot with distributor Excel first.
- [ ] P2.3 `TallyAdapter` (Daybook CSV/XML → `SaleService::create`), then Sage/QB. Edge-agent spec: folder watcher + AES spool + POST to `/fiscal/*`.
- [ ] P2.4 Close 2 paying pilots (10-branch distributor = ~UGX 185M yr-1). Collect testimonial + penalty-avoided metric.
- **Gate:** 2 live paying enterprises, stock variance <1%, import error rate measured. Price/contract template signed off.

### Phase 3  -  Hardware (Weeks 15-20, only if Phase 2 pays)
- [ ] P3.1 Certify white-label Android terminal + thermal print (FDN/QR). 30-day field test.
- [ ] P3.2 Harden offline spool (7-day), battery + network drop tests. Firmware OTA plan.
- **Gate:** zero cache-loss in pilot or defer V2 custom board.

### Phase 4  -  Multi-country (Q3)
- [ ] P4.1 Full `KeEtimsDriver` (KE 16% + eTIMS auth). Western Kenya pilot.
- [ ] P4.2 TZ/RW stubs → full as demand appears. SOM: 250 clients UGX 4.5B/yr.

**Global rules every Phase:** additive migrations only, `migrate --force` on server, `git add <exact paths>`, Vera before commit, Quill docs always, stand-up before inventory/sync/auth work, failure-state review (validation/auth/duplicate/rollback/retry/offline).

In [ ]:
# Phase tracker  -  execute to print current checklist (update done=[...] as we go)
phases = {
    "P0 validate": ["sandbox creds", "spike saveSales", "2 interviews", "ADR headless"],
    "P1 fiscal core": ["credentials+documents tables", "crypto service", "real EfrisClient", "driver interface", "headless routes", "FE badge+settings", "1-branch pilot"],
    "P2 middleware": ["stock recon", "excel import", "tally/sage adapters", "2 paying pilots"],
    "P3 hardware": ["white-label terminal", "7-day spool test"],
    "P4 multi-country": ["eTIMS driver", "TZ/RW stubs"],
}
done = []  # e.g. done = ["sandbox creds"]
for ph, tasks in phases.items():
    print(f"\n{ph.upper()}")
    for t in tasks:
        mark = "[x]" if t in done else "[ ]"
        print(f"  {mark} {t}")
print("\nGate rule: no next Phase until Vera green + Oscar approval.")

## 12. Verification + deploy gates

- **Every handoff:** `composer vera:fast` (BE: `php -l` changed files + logic) and FE `npm run vera:fast` + `npx tsc --noEmit -p tsconfig.app.json`. Extended only on triggers: new migration → `migrate --pretend`; new route → route file lint; matching test → `test --filter=<Name>`.
- **Before any staging/prod touch:** read `/DEPLOYMENT.md` in full. Written deployment plan + Oscar approval + timestamped backup + rollback + verification (HTTP 200, JS MIME, asset count, `schedule:list`, log sweep). Never `migrate:fresh/refresh`, never `key:generate`, never `git add -A`.
- **Pilot SLOs:** checkout p95 <2s offline or online; fiscalized <5min when online; zero sale loss; stock variance <1%; failed fiscal always retryable + visible.

## 13. Risks

| Risk | Mitigation |
|---|---|---|
| URA spec drift / downtime | Driver interface + versioned payload log in `fiscal_documents`; queue absorbs outages |
| Stock mismatch penalties | Recon module + manager-approve adjustments before push |
| Legacy formats vary wildly | Excel first (universal), then one adapter at a time with batch error reports |
| Hardware firmware crashes | Defer custom board; white-label + our app first with kill criteria |
| Scope creep to microservice too early | ADR lock + extract only on >50 tenants / compliance demand |

## 14. Next actions for Oscar

1. Approve modular-monolith + Phase 0 (sandbox + 2 interviews).
2. Assign pilot targets (Mukwano vs Roofings  -  which intro first?).
3. Authorize Quill to file ADR + open P0 branch.

Once approved, Mike runs cross-stack stand-up (Sage+Blue+Atlas+Gauge+Nora) then Rex spikes `saveSales` signing. ✅